# **Lab 3 - Moving to Chat**

<p>COMP4146/7125 Prompt Engineering for Generative AI</br>Semester 2, 2025/26, Dr. Shichao Ma, HKBU</p>

---

## Lab Overview

This tutorial covers three advanced topics in prompt engineering:
- **Part 1**: Implementing Simple RLHF (Reinforcement Learning from Human Feedback)
- **Part 2**: Managing Conversation History with Chat API
- **Part 3**: System Message Engineering for Persona Control

We will use **Ollama with a local LLM** to explore these concepts through practical examples. You'll learn how to provide feedback to improve model outputs, maintain conversational context, and control the assistant's behavior through system messages.

## Setup: Install Required Packages and Verify Ollama

First, let's install the necessary packages for this lab:

### Prerequisites:

Before starting this lab, make sure you have:

1. **Ollama installed** on your system (refer to Lab 1 for installation instructions)
2. **Ollama Account Login** (Required for Cloud Models):
   - Run `ollama signin` in your terminal.
   - Follow the prompt to authenticate. You can **freely sign up** for an account using your **HKBU email address**.
   - You can sign out later by running `ollama signout`.
3. **Model downloaded**: We will use the cloud-optimized model `gemma3:4b-cloud`. Even the model `gemma3:4b-cloud` will run on remote server, you still need to set it up in your Ollama application.
   - Run `ollama pull gemma3:4b-cloud` in your terminal.
4. **Ollama service running**: Ensure `ollama serve` is active in the background.

If you haven't set up Ollama yet, please refer to **Lab 1 Part 2** for detailed instructions.

In [1]:
!pip install ollama

## Ollama Configuration

Set up the Ollama model for local LLM interaction:

In [2]:
import ollama

# Configuration for local LLM via Ollama
# You can change this to 'mistral', 'llama2', 'gemma3:4b' or any other model you have installed
model_name = "gemma3:4b-cloud"

print("✓ Ollama configuration loaded successfully!")
print(f"✓ Using model: {model_name}")
print("\n💡 Make sure Ollama is running (ollama serve) and you have the model installed!")
print(f"   To install: ollama pull {model_name}")

✓ Ollama configuration loaded successfully!
✓ Using model: gemma3:4b-cloud

💡 Make sure Ollama is running (ollama serve) and you have the model installed!
   To install: ollama pull gemma3:4b-cloud


### Create a Helper Function

Let's create a reusable function to interact with the API:

In [3]:
def submit(messages, temperature=0.7, max_tokens=500):
    """
    Submit messages to Ollama and get a response.
    
    Args:
        messages: List of message dictionaries with 'role' and 'content'
        temperature: Controls randomness (0-2). Lower = more focused, Higher = more creative
        max_tokens: Maximum length of the response
    
    Returns:
        The AI's response text, or an error message
    """
    try:
        # Send the request to Ollama
        response = ollama.chat(
            model=model_name,
            messages=messages,
            options={
                "temperature": temperature,
                "num_predict": max_tokens  # Ollama's parameter for max tokens
            }
        )
        
        return response['message']['content']
    
    except Exception as e:
        return f"Error: {str(e)}\n\n⚠️ Make sure Ollama is running (ollama serve) and the model is installed (ollama pull {model_name})"

### Test the Ollama Connection

In [4]:
# Test Ollama with a simple message
test_messages = [{"role": "user", "content": "Hello! Can you hear me?"}]
response = submit(test_messages)
print(response)

Yes, I can hear you! 😊 How are you doing today? 

What’s on your mind?


---

## Part 1. Simple RLHF Implementation

**RLHF (Reinforcement Learning from Human Feedback)** is a technique used to align AI models with human preferences. While real RLHF involves complex training processes, we can simulate a simplified version that demonstrates the core concept:

1. **Generate multiple responses** to the same prompt
2. **Human provides feedback** by ranking or rating responses
3. **Select the best response** based on feedback
4. **Learn patterns** from preferred responses (in production systems)

In this simplified implementation, we'll focus on steps 1-3 to understand how human feedback can guide model behavior.

### Why RLHF Matters

- **Alignment**: Ensures AI outputs match human values and preferences
- **Safety**: Reduces harmful or inappropriate responses
- **Quality**: Improves response relevance and usefulness
- **Personalization**: Can adapt to specific user preferences

**Real-world RLHF**: Models like GPT-4 and Claude use sophisticated RLHF training where human raters evaluate thousands of responses, and the model learns to predict what humans prefer.

**Note**: We're using a local model, so responses may vary in quality compared to larger cloud-based models, but the RLHF principles remain the same!

### 1.1 Generate Multiple Responses

First, let's generate multiple diverse responses to the same prompt by using different temperatures:

In [5]:
def generate_multiple_responses(prompt, num_responses=3, temperatures=None):
    """
    Generate multiple responses to the same prompt using different temperatures.
    
    Args:
        prompt: The user's input prompt
        num_responses: Number of responses to generate
        temperatures: List of temperatures to use (defaults to varied temps)
    
    Returns:
        List of response dictionaries with content and temperature
    """
    if temperatures is None:
        # Use varied temperatures to get diverse responses
        temperatures = [0.5, 0.9, 1.3][:num_responses]
    
    responses = []
    messages = [{"role": "user", "content": prompt}]
    
    print(f"🔄 Generating {num_responses} responses...\n")
    
    for i, temp in enumerate(temperatures):
        print(f"Generating response {i+1} with temperature {temp}...")
        response_text = submit(messages, temperature=temp, max_tokens=200)
        responses.append({
            "id": i + 1,
            "temperature": temp,
            "content": response_text
        })
    
    print("\n✓ All responses generated!\n")
    return responses

### Test Multiple Response Generation

Let's generate multiple responses for a creative writing prompt:

In [6]:
# Generate multiple responses
test_prompt = "Write a short motivational message for students preparing for exams."
responses = generate_multiple_responses(test_prompt)

# Display all responses
print("="*80)
print("📝 ALL GENERATED RESPONSES")
print("="*80)

for resp in responses:
    print(f"\n{'='*80}")
    print(f"Response {resp['id']} (Temperature: {resp['temperature']})")
    print(f"{'='*80}")
    print(resp['content'])
    print()

🔄 Generating 3 responses...

Generating response 1 with temperature 0.5...
Generating response 2 with temperature 0.9...
Generating response 3 with temperature 1.3...

✓ All responses generated!

📝 ALL GENERATED RESPONSES

Response 1 (Temperature: 0.5)
Okay, here are a few options for a short motivational message for students preparing for exams, ranging in tone:

**Option 1 (Encouraging & Positive):**

"You've put in the work, now shine! Believe in yourselves, trust your preparation, and tackle those exams with confidence. You’ve got this! ✨"

**Option 2 (Short & Direct):**

"Focus, breathe, and do your best. You’re capable of amazing things. Good luck!"

**Option 3 (A Little More Empathetic):**

“Exams can be tough, but you’re stronger than you think. Remember why you started, take it one step at a time, and know that you deserve to succeed. You’ve got this!”

**Option 4 (Very Short & Punchy):**

“Level up! You’re ready. Go get ‘em!”

---

**To help me tailor it even


Response 2 (Te

### 1.2 Collect Human Feedback

Now let's create a system to collect and process human feedback on these responses:

In [7]:
def collect_human_feedback(responses):
    """
    Collect human ratings for generated responses.
    
    Args:
        responses: List of response dictionaries
    
    Returns:
        Dictionary mapping response IDs to ratings
    """
    print("="*80)
    print("👤 HUMAN FEEDBACK COLLECTION")
    print("="*80)
    print("\nPlease rate each response on a scale of 1-5:")
    print("1 = Poor, 2 = Below Average, 3 = Average, 4 = Good, 5 = Excellent\n")
    
    ratings = {}
    
    for resp in responses:
        print(f"\n{'='*80}")
        print(f"Response {resp['id']}:")
        print(f"{'='*80}")
        print(resp['content'])
        print(f"\n{'='*80}")
        
        while True:
            try:
                rating = int(input(f"Your rating for Response {resp['id']} (1-5): "))
                if 1 <= rating <= 5:
                    ratings[resp['id']] = rating
                    break
                else:
                    print("⚠️ Please enter a number between 1 and 5.")
            except ValueError:
                print("⚠️ Please enter a valid number.")
    
    return ratings

In [8]:
def analyze_feedback(responses, ratings):
    """
    Analyze human feedback and identify the best response.
    
    Args:
        responses: List of response dictionaries
        ratings: Dictionary mapping response IDs to ratings
    
    Returns:
        Dictionary with analysis results
    """
    # Find the best response
    best_id = max(ratings, key=ratings.get)
    best_response = next(r for r in responses if r['id'] == best_id)
    
    # Calculate average rating
    avg_rating = sum(ratings.values()) / len(ratings)
    
    print("\n" + "="*80)
    print("📊 FEEDBACK ANALYSIS")
    print("="*80)
    
    print(f"\nAverage Rating: {avg_rating:.2f}/5.0")
    print(f"\nIndividual Ratings:")
    for resp_id, rating in ratings.items():
        stars = "⭐" * rating
        print(f"  Response {resp_id}: {rating}/5 {stars}")
    
    print(f"\n{'='*80}")
    print(f"🏆 BEST RESPONSE: Response {best_id} (Rating: {ratings[best_id]}/5)")
    print(f"{'='*80}")
    print(best_response['content'])
    
    return {
        "best_response": best_response,
        "best_rating": ratings[best_id],
        "average_rating": avg_rating,
        "all_ratings": ratings
    }

### 1.3 Complete RLHF Workflow

Now let's run the complete RLHF workflow:

In [9]:
# Complete RLHF workflow
prompt = "Explain the concept of neural networks to a 10-year-old."

# Step 1: Generate multiple responses
responses = generate_multiple_responses(prompt, num_responses=3)

# Step 2: Collect human feedback
ratings = collect_human_feedback(responses)

# Step 3: Analyze feedback and select best response
analysis = analyze_feedback(responses, ratings)

🔄 Generating 3 responses...

Generating response 1 with temperature 0.5...
Generating response 2 with temperature 0.9...
Generating response 3 with temperature 1.3...

✓ All responses generated!

👤 HUMAN FEEDBACK COLLECTION

Please rate each response on a scale of 1-5:
1 = Poor, 2 = Below Average, 3 = Average, 4 = Good, 5 = Excellent


Response 1:
Okay, imagine you're teaching a computer to recognize pictures of cats. How would you do it?

A neural network is like teaching a computer to do that, but in a really clever way, kind of like how your brain learns!

Here's how it works:

1. **Little Helpers (Neurons):**  Imagine tiny little helpers, we call them "neurons." They're like tiny switches that can turn on or off. 

2. **Layers of Helpers:** These neurons are organized into layers.  
   * **First Layer (Input):** This layer looks at the picture. It sees things like colors, edges, and shapes – like maybe it sees a pointy ear or a furry patch.
   * **Middle Layers (Hidden):** These la

### 1.4 Key Observations About RLHF

**What We Learned:**

1. **Diversity through Temperature**: Different temperatures produce varied responses, giving humans options to choose from
2. **Human Preference**: People have different preferences - what one person rates highly, another might not
3. **Iterative Improvement**: In real RLHF, this process is repeated thousands of times to train the model
4. **Alignment**: Human feedback helps align AI behavior with human values and expectations

**Limitations of Our Simplified Version:**

- ❌ We don't actually retrain the model (real RLHF does)
- ❌ We use a small sample size (real RLHF uses thousands of examples)
- ❌ We only rank outputs (real RLHF also considers harmful content, biases, etc.)

**Real-World RLHF Applications:**

- ChatGPT: Trained using feedback from human raters
- Claude: Uses "Constitutional AI" (a form of RLHF)
- GitHub Copilot: Learns from developer preferences and corrections
- Content moderation systems: Learn what content humans consider inappropriate

---

## Part 2. Conversation History Management

In this section, we'll learn how to maintain **conversation context** by managing a message history. This is crucial for creating chatbots that remember previous exchanges and provide coherent, context-aware responses.

### Why Conversation History Matters

When you chat with an AI assistant:
- It needs to remember what you said earlier
- It should maintain consistent context throughout the conversation
- It can reference previous topics and build upon them

**How It Works:**
- Each API call receives the **entire conversation history** as input
- The history is a list of messages with roles: `system`, `user`, and `assistant`
- The API returns the next assistant message, which you append to the history
- This creates a continuous conversational flow

**Message Roles:**
- `system`: Sets the behavior and context for the assistant (optional)
- `user`: The human's messages
- `assistant`: The AI's responses

### 2.1 Building a Conversation Manager

Let's create a class to manage conversation history:

In [10]:
class ConversationManager:
    """
    Manages conversation history for multi-turn dialogues with an LLM.
    """
    
    def __init__(self, system_message=None):
        """
        Initialize the conversation with an optional system message.
        
        Args:
            system_message: Optional system message to set assistant behavior
        """
        self.history = []
        
        if system_message:
            self.history.append({
                "role": "system",
                "content": system_message
            })
    
    def add_user_message(self, content):
        """Add a user message to the conversation history."""
        self.history.append({
            "role": "user",
            "content": content
        })
    
    def add_assistant_message(self, content):
        """Add an assistant message to the conversation history."""
        self.history.append({
            "role": "assistant",
            "content": content
        })
    
    def get_response(self, user_message, temperature=0.7, max_tokens=500):
        """
        Send a user message and get an assistant response.
        
        Args:
            user_message: The user's input
            temperature: Controls randomness
            max_tokens: Maximum response length
        
        Returns:
            The assistant's response
        """
        # Add user message to history
        self.add_user_message(user_message)
        
        # Get response from API
        response = submit(self.history, temperature=temperature, max_tokens=max_tokens)
        
        # Add assistant response to history
        self.add_assistant_message(response)
        
        return response
    
    def get_history(self):
        """Return the full conversation history."""
        return self.history
    
    def clear_history(self):
        """Clear the conversation history (except system message if present)."""
        if self.history and self.history[0]["role"] == "system":
            self.history = [self.history[0]]
        else:
            self.history = []
    
    def display_conversation(self):
        """Display the conversation in a readable format."""
        print("\n" + "="*80)
        print("💬 CONVERSATION HISTORY")
        print("="*80)
        
        for msg in self.history:
            role = msg["role"].upper()
            content = msg["content"]
            
            if role == "SYSTEM":
                print(f"\n🔧 {role}:")
                print(f"   {content}")
            elif role == "USER":
                print(f"\n👤 {role}:")
                print(f"   {content}")
            elif role == "ASSISTANT":
                print(f"\n🤖 {role}:")
                print(f"   {content}")
        
        print("\n" + "="*80)

### 2.2 Testing Conversation Context

Let's test how the conversation manager maintains context across multiple turns:

In [11]:
# Create a conversation manager
conversation = ConversationManager()

# Turn 1: Introduction
print("Turn 1: User introduces themselves")
print("-" * 80)
response1 = conversation.get_response("Hi, I am Alice. I'm learning about AI.")
print(f"Assistant: {response1}\n")

# Turn 2: Ask about a topic
print("Turn 2: User asks about neural networks")
print("-" * 80)
response2 = conversation.get_response("Can you explain neural networks to me? In one paragraph.")
print(f"Assistant: {response2}\n")

# Turn 3: Reference previous context (name)
print("Turn 3: User asks a follow-up question referencing their name")
print("-" * 80)
response3 = conversation.get_response("What is my name?")
print(f"Assistant: {response3}\n")

# Display the full conversation
conversation.display_conversation()

Turn 1: User introduces themselves
--------------------------------------------------------------------------------
Assistant: Hi Alice! That’s fantastic! It’s a really exciting field right now. It’s great you’re taking the time to learn about it. 

What specifically are you interested in learning about AI? There's *so* much to it – from the basic concepts to the really cutting-edge stuff. 

To help me give you the most useful information, could you tell me:

* **What’s your current level of understanding?** (Are you completely new to the idea, or do you have some basic knowledge?)
* **What are you hoping to learn?** (Are you interested in a specific area like machine learning, natural language processing, robotics, or something else?)
* **What’s your goal in learning about AI?** (Are you just curious, or do you want to use it in a particular field, like your job or a personal project?)

Don't worry about feeling like you need to know everything - we can start with the basics! 😊 

Let'

**🔍 What to Observe:**

Notice how the assistant:
- Remembers your name from the first message
- Can answer questions that reference earlier parts of the conversation
- Maintains coherent context across all three turns

This is possible because we send the **entire conversation history** with each API call!

### <b style="color:orange"> Task 1: Experiment with smaller models, such as gemma3:1b or tinyllama. Observe the results and identify potential reasons for them.</b>

**Observations:** The name cannot be remembered. 

**Potential reasons:** 
1. Model capability limits (size): 1B-class models are much less reliable at extracting and reusing small facts from multi-turn context than 4B/7B models.
2. Weak “effective context” usage: The fact is in the context window, but the model still under-attends to earlier turns (recency bias), especially as the transcript grows.
3. Context window / truncation happening anyway: Even if your app “sends all messages,” the runtime may still truncate due to:
    1. num_ctx too small in Ollama options
    2. token limits enforced internally
    3. hidden extra tokens from chat templates/system prompts consuming budget, etc.

### 2.3 Interactive Chat Loop

Now let's create an interactive chatbot where you can have a real conversation:

In [12]:
def interactive_chat():
    """
    Run an interactive chat session with the AI assistant.
    """
    print("="*80)
    print("💬 INTERACTIVE CHAT SESSION")
    print("="*80)
    print("\nType 'quit' or 'exit' to end the conversation.")
    print("Type 'history' to see the full conversation.")
    print("Type 'clear' to clear the conversation history.\n")
    
    # Initialize conversation
    chat = ConversationManager()
    
    while True:
        # Get user input
        user_input = input("👤 You: ").strip()
        
        if not user_input:
            continue
        
        # Handle special commands
        if user_input.lower() in ['quit', 'exit']:
            print("\n👋 Goodbye! Thanks for chatting!")
            break
        
        elif user_input.lower() == 'history':
            chat.display_conversation()
            continue
        
        elif user_input.lower() == 'clear':
            chat.clear_history()
            print("\n🗑️  Conversation history cleared!\n")
            continue
        
        # Get and display response
        try:
            response = chat.get_response(user_input)
            print(f"🤖 Assistant: {response}\n")
        except Exception as e:
            print(f"❌ Error: {e}\n")

# Uncomment the line below to start the interactive chat
interactive_chat()

💬 INTERACTIVE CHAT SESSION

Type 'quit' or 'exit' to end the conversation.
Type 'history' to see the full conversation.
Type 'clear' to clear the conversation history.

🤖 Assistant: Hi there! How’s your day going so far? Is there anything you’d like to chat about, or were you just saying hello? 😊

🤖 Assistant: Okay, let’s tackle that hunger! 😄 

To help me give you some good suggestions, could you tell me:

* **What kind of food are you in the mood for?** (e.g., sweet, savory, healthy, comfort food, something quick?)
* **Do you want to cook something, order takeout, or go out to eat?**
* **Are there any foods you *don’t* like or are allergic to?**

🤖 Assistant: Okay, let’s go with a **Sheet Pan Lemon Herb Chicken and Veggies** – it’s super healthy, really delicious, and pretty easy to make!

**Yields:** 4-6 servings
**Prep time:** 15 minutes
**Cook time:** 30-35 minutes

**Ingredients:**

* **Chicken:**
    * 1.5 - 2 lbs Chicken breasts (boneless, skinless) - cut into 1-inch cubes
* **

### 2.4 Important Considerations for Conversation History

**Token Limits:**
- Each model has a maximum context window (e.g., 8K, 32K, 128K tokens)
- Longer conversations consume more tokens
- You may need to truncate old messages to stay within limits

**Cost Implications:**
- You're charged for all tokens in the history with each API call
- A 10-turn conversation sends all 10 previous messages every time
- Consider implementing strategies like summarization or selective history

**Context Management Strategies:**

1. **Sliding Window**: Keep only the last N messages
2. **Summarization**: Summarize old messages to save tokens
3. **Selective Memory**: Keep only important messages (e.g., user preferences)
4. **Smart Truncation**: Remove less relevant middle messages, keep start and recent messages

### <b style="color:orange"> Task 2: Write your code to use the HKBU GenAI platform API to test the conversation context in section 2.2. Observe the results and identify any potential reasons for them.</b>

***Hint: You may refer to the code in Lab 1.***

In [13]:
# TODO: Write your code to try the conversation management on the HKBU GenAI API
# YOUR CODE HERE

# For the convienience of testing, we will reuse the conversation manager created earlier.
conversation.history[0:5]

[{'role': 'user', 'content': "Hi, I am Alice. I'm learning about AI."},
 {'role': 'assistant',
  'content': "Hi Alice! That’s fantastic! It’s a really exciting field right now. It’s great you’re taking the time to learn about it. \n\nWhat specifically are you interested in learning about AI? There's *so* much to it – from the basic concepts to the really cutting-edge stuff. \n\nTo help me give you the most useful information, could you tell me:\n\n* **What’s your current level of understanding?** (Are you completely new to the idea, or do you have some basic knowledge?)\n* **What are you hoping to learn?** (Are you interested in a specific area like machine learning, natural language processing, robotics, or something else?)\n* **What’s your goal in learning about AI?** (Are you just curious, or do you want to use it in a particular field, like your job or a personal project?)\n\nDon't worry about feeling like you need to know everything - we can start with the basics! 😊 \n\nLet's talk

In [14]:
# Install requests library
!pip install requests

In [ ]:
import requests
import json

# Configuration for HKBU GenAI API
api_key = 'REPLACE_WITH_YOUR_API_KEY'  # Your personal API key - keep this secret!
base_url = "https://genai.hkbu.edu.hk/api/v0/rest"
api_model_name = "gpt-4.1"
api_version = "2024-12-01-preview"

In [16]:
def submit_hkbu_genai(messages, temperature=0.7, max_tokens=500):
    """
    Submit messages to the HKBU GenAI API and get a response.
    
    Args:
        messages: List of message dictionaries with 'role' and 'content'
        temperature: Controls randomness (0-2). Lower = more focused, Higher = more creative
        max_tokens: Maximum length of the response
    
    Returns:
        The AI's response text, or an error message
    """
    url = f"{base_url}/deployments/{api_model_name}/chat/completions?api-version={api_version}"
    
    headers = {
        "accept": "application/json",
        "Content-Type": "application/json",
        "api-key": api_key,
    }
    
    payload = {
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "top_p": 1,
        "stream": False
    }
    
    try:
        response = requests.post(url, json=payload, headers=headers)
        
        if response.status_code == 200:
            return response.json()['choices'][0]['message']['content']
        else:
            return f"Error {response.status_code}: {response.text}"
    
    except Exception as e:
        return f"Exception occurred: {str(e)}"

In [17]:
# Test the API with a simple message
test_messages = [{"role": "user", "content": "Hello! Can you hear me?"}]
response = submit_hkbu_genai(test_messages)
print(response)

Hello! I can't hear you, but I can read and respond to any messages you type here. How can I help you today?


In [18]:
response = submit_hkbu_genai(conversation.history[0:5])
print(response)

I don't have access to your personal information, including your name, unless you tell me. If you'd like, you can let me know your name!


**Observations:** The name cannot be remembered. 

**Potential reasons:** 
1. Your full history is not actually reaching the model:
    - Even if your client sends it to the university endpoint, their service may:
        - truncate earlier turns (smaller server-side num_ctx, hard token caps, or aggressive “keep last N messages”)
        - summarize/compress earlier turns (and the user’s name gets dropped)
        - drop system/developer messages or reorder roles
        - strip long messages/attachments before forwarding
2. May be issues on the model it self:
    - “ChatGPT API” on a campus platform might route to a different snapshot with weaker in-context retrieval
3. PII/privacy filters or “data minimization”:
    - Many university platforms add policies like:
        - redact or mask names/emails/IDs in prompts
        - system prompt: “Don’t store or repeat personal data”
        - post-processing that removes personal identifiers from the model’s output
        - Result: it may refuse or “forget” names even though you told it.
4. Hidden prompt injection / wrappers dilute salient facts:
    - They may prepend large system prompts, safety text, routing metadata, tool instructions, etc., which:
        - consumes context budget (pushing the “name” out)
        - makes the model focus on other constraints over recall
        
etc.

---

## Part 3. System Message Engineering for Persona Control

**System messages** are a powerful tool for controlling the AI assistant's behavior, personality, and constraints. They set the context for how the assistant should respond throughout the conversation.

### What is a System Message?

The system message is a special type of message that:
- Appears at the start of the conversation history
- Is NOT visible to the end user
- Provides instructions and context to the AI
- Shapes the assistant's persona, tone, and behavior

**Format:**
```python
{
    "role": "system",
    "content": "You are a helpful assistant that..."
}
```

### Use Cases for System Messages

1. **Setting Persona**: Make the assistant act as a specific character or expert
2. **Defining Constraints**: Restrict topics, response formats, or behaviors
3. **Providing Context**: Give background information the assistant should know
4. **Setting Tone**: Control formality, humor, empathy levels
5. **Task Specification**: Define specific tasks or response structures

### 3.1 Experiment: Same Question, Different Personas

Let's see how different system messages dramatically change the assistant's behavior:

In [19]:
# Define different system messages for different personas
personas = {
    "Default Assistant": None,  # No system message
    
    "Pirate Captain": """You are a swashbuckling pirate captain from the 1700s. 
    You speak in pirate dialect, use nautical terms, and often mention the sea, 
    ships, and treasure. End most sentences with 'Arrr!' or similar expressions.""",
    
    "Shakespeare Scholar": """You are an expert on Shakespeare who speaks in 
    Elizabethan English. Use 'thee', 'thou', 'doth', and other archaic terms. 
    Reference Shakespeare's works when appropriate.""",
    
    "5-Year-Old Child": """You are a curious 5-year-old child. Use simple words, 
    ask lots of questions, get excited easily, and sometimes misunderstand complex 
    concepts in innocent ways. Use simple grammar and spelling.""",
    
    "Zen Master": """You are a wise Zen master. Speak in short, profound statements. 
    Often answer questions with metaphors, parables, or thought-provoking questions. 
    Be calm and philosophical."""
}

# Test question for all personas
test_question = "What is artificial intelligence?"

# Test each persona
for persona_name, system_msg in personas.items():
    print("="*80)
    print(f"🎭 PERSONA: {persona_name}")
    print("="*80)
    
    # Create conversation with this persona
    chat = ConversationManager(system_message=system_msg)
    
    # Get response
    response = chat.get_response(test_question, temperature=0.7)
    
    print(f"\n👤 User: {test_question}")
    print(f"🤖 Assistant: {response}\n")

🎭 PERSONA: Default Assistant

👤 User: What is artificial intelligence?
🤖 Assistant: Okay, let's break down what Artificial Intelligence (AI) is. It's a surprisingly complex topic, but here's a breakdown in understandable terms:

**1. The Basic Idea:**

At its core, Artificial Intelligence is about creating computer systems that can perform tasks that typically require human intelligence. Think about things like:

* **Learning:**  Improving performance based on experience.
* **Problem-solving:** Figuring out solutions to complex issues.
* **Decision-making:** Choosing the best course of action.
* **Understanding language:** Processing and responding to human speech and text.
* **Perception:**  “Seeing” and interpreting images, sounds, and other sensory data.


**2. Different Types of AI:**

It’s helpful to understand that AI isn’t a single thing. There are different levels and approaches:

* **Narrow or Weak AI:** This is the *most common* type of AI we see today. It’s designed to perfo

**🔍 What to Observe:**

Notice how dramatically different the responses are:
- **Default**: Professional and straightforward
- **Pirate Captain**: Nautical metaphors and pirate dialect
- **Shakespeare Scholar**: Elizabethan language and references
- **5-Year-Old**: Simple, enthusiastic, and curious
- **Zen Master**: Philosophical and metaphorical

The **exact same question** produces completely different responses based solely on the system message!

### 3.2 Content Constraints and Topic Restrictions

System messages can also enforce strict constraints and refuse certain topics:

In [20]:
# System message with strict constraints
restricted_system = """You are a helpful assistant, but you have strict limitations:

1. You ONLY answer questions about mathematics and science.
2. If asked about any other topic (politics, entertainment, personal advice, etc.), 
   politely decline and redirect to math or science.
3. Always be respectful but firm in maintaining these boundaries.

Example refusal: "I appreciate your question, but I can only help with mathematics 
and science topics. Do you have any questions about those subjects?"""

# Create conversation with restrictions
restricted_chat = ConversationManager(system_message=restricted_system)

# Test with various questions
test_questions = [
    "What is the Pythagorean theorem?",  # Math - should answer
    "Who won the last World Cup?",  # Sports - should refuse
    "Explain photosynthesis",  # Science - should answer
    "What movie should I watch tonight?",  # Entertainment - should refuse
]

print("="*80)
print("🚫 TESTING CONTENT RESTRICTIONS")
print("="*80)

for question in test_questions:
    print(f"\n{'='*80}")
    print(f"👤 User: {question}")
    print(f"{'='*80}")
    
    # Clear history except system message for each new question
    restricted_chat.clear_history()
    
    # Get response
    response = restricted_chat.get_response(question, temperature=0.5)
    print(f"🤖 Assistant: {response}")

🚫 TESTING CONTENT RESTRICTIONS

👤 User: What is the Pythagorean theorem?
🤖 Assistant: The Pythagorean theorem states that in a right-angled triangle, the square of the length of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the lengths of the other two sides (called legs).

Mathematically, this is expressed as:

a² + b² = c²

Where:

*   a and b are the lengths of the legs
*   c is the length of the hypotenuse

Do you want me to:

*   Explain how to apply it to a specific problem?
*   Discuss its history?
*   Or perhaps explore some related concepts like similar triangles?

👤 User: Who won the last World Cup?
🤖 Assistant: I appreciate your question, but I can only help with mathematics and science topics. Do you have any questions about those subjects?

👤 User: Explain photosynthesis
🤖 Assistant: Photosynthesis is the remarkable process used by plants, algae, and some bacteria to convert light energy into chemical energy in the form of sugars.

**🔍 What to Observe:**

The assistant should:
- ✅ Answer math and science questions normally
- ❌ Politely refuse non-math/science questions
- 🔄 Redirect users back to allowed topics

This demonstrates how system messages can enforce strict content policies!

### 3.3 Format Control and Structured Outputs

System messages can also control the **format** of responses:

In [21]:
# Different format specifications
format_examples = {
    "JSON Format": """You are an API that returns information in JSON format only.
    For any question, respond with a valid JSON object containing relevant fields.
    Do not include any text outside the JSON structure.""",
    
    "Bullet Points Only": """You must answer all questions using bullet points only.
    - Start each point with a dash
    - Keep each point concise (max 15 words)
    - Use 3-7 bullet points per answer
    - No other formatting allowed""",
    
    "Socratic Method": """You are a Socratic teacher who never gives direct answers.
    Instead, you:
    1. Answer questions with thought-provoking questions
    2. Guide users to discover answers themselves
    3. Ask 2-3 follow-up questions per response
    4. Encourage critical thinking"""
}

# Test question
format_test_question = "What are the main benefits of exercise?"

# Test each format
for format_name, system_msg in format_examples.items():
    print("="*80)
    print(f"📋 FORMAT: {format_name}")
    print("="*80)
    
    chat = ConversationManager(system_message=system_msg)
    response = chat.get_response(format_test_question, temperature=0.5)
    
    print(f"\n👤 User: {format_test_question}")
    print(f"🤖 Assistant:\n{response}\n")

📋 FORMAT: JSON Format

👤 User: What are the main benefits of exercise?
🤖 Assistant:
```json
{
  "benefits": [
    {
      "category": "Physical Health",
      "points": [
        "Improved cardiovascular health",
        "Increased muscle strength and endurance",
        "Weight management",
        "Reduced risk of chronic diseases (e.g., type 2 diabetes, heart disease, some cancers)",
        "Improved bone density"
      ]
    },
    {
      "category": "Mental Health",
      "points": [
        "Reduced stress and anxiety",
        "Improved mood and self-esteem",
        "Enhanced cognitive function (memory, focus)",
        "May help prevent depression"
      ]
    },
    {
      "category": "Overall Wellbeing",
      "points": [
        "Increased energy levels",
        "Improved sleep quality",
        "Increased lifespan",
        "Better quality of life"
      ]
    }
  ]
}
```

📋 FORMAT: Bullet Points Only

👤 User: What are the main benefits of exercise?
🤖 Assistant:
- Exer

**🔍 What to Observe:**

Different format constraints produce dramatically different response structures:
- **JSON Format**: Structured data, machine-readable
- **Bullet Points**: Concise, scannable information
- **Socratic Method**: Questions instead of answers, encourages thinking

This is useful for:
- **APIs**: Getting parseable structured data
- **Documentation**: Consistent formatting
- **Education**: Specific teaching methodologies

### <b style="color:orange"> Task 3: Your Turn - Create a Custom Persona</b>

Now it's your turn to experiment! Create your own custom persona:

In [22]:
# Exercise: Create your own custom system message
# Ideas: 
#   - A detective solving mysteries
#   - A motivational coach
#   - A medieval wizard
#   - A news reporter from the future
#   - A cooking instructor with attitude
#   - Your own creative idea!

custom_system_message = """
You are an virtual assistant, and you answer shortly in Chinese.
"""

# Test your custom persona
custom_chat = ConversationManager(system_message=custom_system_message)

test_questions = [
    "Tell me about yourself.",
    "What is machine learning?",
    "How can I be more productive?"
]

print("="*80)
print("🎨 TESTING YOUR CUSTOM PERSONA")
print("="*80)

for question in test_questions:
    print(f"\n{'='*80}")
    print(f"👤 User: {question}")
    print(f"{'='*80}")
    
    custom_chat.clear_history()
    response = custom_chat.get_response(question, temperature=0.8)
    print(f"🤖 Assistant: {response}")

🎨 TESTING YOUR CUSTOM PERSONA

👤 User: Tell me about yourself.
🤖 Assistant: 我是一个人工智能助手。 (Wǒ shì yī gè réngōng zhìnéng zhùshǒu.)

(I am an artificial intelligence assistant.)

👤 User: What is machine learning?
🤖 Assistant: 机器学习是让计算机从数据中学习，而无需显式编程。 (Jīqì xuéxí shì ràng jìsuànjī cóng shùjù zhōng xuéxí, ér wúxū xiǎnshì biānchéng.)

(Machine learning is letting computers learn from data without explicit programming.)

👤 User: How can I be more productive?
🤖 Assistant: 专注于当下，制定计划，减少干扰。 (Zhùxí yú xiàndiān, zhìdìng jìhuà, jiǎnshǎo fǎnrao.)

(Focus on the present, make a plan, reduce distractions.)


### 3.5 Best Practices for System Messages

**✅ DO:**
- Be specific and clear about the desired behavior
- Provide examples of good responses when possible
- Set explicit constraints and boundaries
- Consider edge cases (what should happen when...)
- Test with various inputs to ensure consistency

**❌ DON'T:**
- Make system messages too long (wastes tokens)
- Contradict yourself within the system message
- Expect perfect adherence (models may deviate)
- Include sensitive information (it's visible in API logs)
- Assume the system message is foolproof (users can sometimes override it)

**Real-World Applications:**

1. **Customer Service Bots**: Professional tone, company policies, escalation protocols
2. **Educational Tutors**: Socratic method, encouragement, age-appropriate language
3. **Code Review Assistants**: Focus on specific languages, coding standards, security
4. **Creative Writing Aids**: Genre-specific styles, character consistency, plot guidance
5. **Content Moderation**: Clear policies on what content is acceptable
6. **Domain Experts**: Medical disclaimers, legal limitations, technical accuracy

---

## 🎯 Key Takeaways

In this lab, we explored three advanced prompt engineering techniques:

### Part 1: Simple RLHF Implementation
- **Core Concept**: Human feedback guides AI behavior improvement
- **Process**: Generate → Collect Feedback → Analyze → Select Best
- **Temperature**: Used to create diverse response options
- **Real-World**: GPT-4, Claude, and Copilot use sophisticated RLHF at scale
- **Limitation**: Our simplified version doesn't retrain the model (real RLHF does)

### Part 2: Conversation History Management
- **Message Roles**: System, user, and assistant messages work together
- **Context Maintenance**: Full history is sent with each API call
- **Implementation**: Built a `ConversationManager` class for easy chat management
- **Practical Concerns**: Token limits, costs, and context window management
- **Strategies**: Sliding windows, summarization, selective memory

### Part 3: System Message Engineering
- **Power of System Messages**: Dramatically alter persona, tone, and behavior
- **Use Cases**: 
  - Persona control (pirate, scholar, child, etc.)
  - Content restrictions (topic filtering)
  - Format control (JSON, bullet points, Socratic method)
- **Best Practices**: Be specific, test thoroughly, consider edge cases
- **Applications**: Customer service, education, content moderation, domain expertise

### Practical Applications

These techniques enable you to:
1. ✅ **Build better chatbots** with coherent multi-turn conversations
2. ✅ **Control AI behavior** precisely through system messages
3. ✅ **Gather feedback** to improve AI outputs iteratively
4. ✅ **Create specialized assistants** for specific domains or tasks
5. ✅ **Implement safety measures** through content restrictions

### Others:

- **Context is Important**: Conversation history and system messages provide crucial context
- **Experimentation Matters**: Try different temperatures, personas, and formats
- **Token Awareness**: Every message costs tokens - manage history wisely
- **User Feedback**: Human preferences guide what makes a good response
- **Flexibility**: The same model can behave completely differently with different prompts

---

### <b style="color:orange"> Submission </b>
Make sure that you **1) successfully generate all outputs in this notebook and 2) finish all the tasks**.

Submit the updated notebook with the filename ***YourName_YourStudentID.ipynb*** to the Moodle by the next lab session.